In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
train_values = pd.read_csv('../data/raw/train_values.csv')
train_labels = pd.read_csv('../data/raw/train_labels.csv')
test_values  = pd.read_csv('../data/raw/test_values.csv')
submission   = pd.read_csv('../data/raw/submission_format.csv')

# Join train features + target
train = train_values.merge(train_labels, on='building_id', how='left')

print(f'Train shape : {train.shape}')
print(f'Test shape  : {test_values.shape}')
train.head()

Train shape : (260601, 40)
Test shape  : (86868, 39)


,building_id,geo_level_1_id,geo_level_2_id,geo_level_3_id,count_floors_pre_eq,age,area_percentage,height_percentage,land_surface_condition,foundation_type,...,has_secondary_use_hotel,has_secondary_use_rental,has_secondary_use_institution,has_secondary_use_school,has_secondary_use_industry,has_secondary_use_health_post,has_secondary_use_gov_office,has_secondary_use_use_police,has_secondary_use_other,damage_grade
0,802906,6,487,12198,2,30,6,5,t,r,...,0,0,0,0,0,0,0,0,0,3
1,28830,8,900,2812,2,10,8,7,o,r,...,0,0,0,0,0,0,0,0,0,2
2,94947,21,363,8973,2,10,5,5,t,r,...,0,0,0,0,0,0,0,0,0,3
3,590882,22,418,10694,2,10,6,5,t,r,...,0,0,0,0,0,0,0,0,0,2
4,201944,11,131,1488,3,30,8,9,t,r,...,0,0,0,0,0,0,0,0,0,3


---
## 1. Tipi di variabili

In [4]:
# Tipi di variabili
dtype_summary = pd.DataFrame({
    'dtype'       : train.dtypes.astype(str),
    'n_unique'    : train.nunique(),
}).sort_values('dtype')

print(dtype_summary.to_string())
print(f"\nTotale colonne  : {train.shape[1]}")
print(f"  • int64        : {(train.dtypes == 'int64').sum()}")
print(f"  • object (str) : {(train.dtypes == 'object').sum()}")

                                         dtype  n_unique
building_id                              int64    260601
has_superstructure_timber                int64         2
has_superstructure_bamboo                int64         2
has_superstructure_rc_non_engineered     int64         2
has_superstructure_rc_engineered         int64         2
has_superstructure_other                 int64         2
count_families                           int64        10
has_secondary_use                        int64         2
has_superstructure_cement_mortar_brick   int64         2
has_secondary_use_agriculture            int64         2
has_secondary_use_rental                 int64         2
has_secondary_use_institution            int64         2
has_secondary_use_school                 int64         2
has_secondary_use_industry               int64         2
has_secondary_use_health_post            int64         2
has_secondary_use_gov_office             int64         2
has_secondary_use_use_police   

In [5]:
# Colonne per ruolo semantico
COL_ID      = ['building_id']
COL_TARGET  = ['damage_grade']

COL_CAT     = ['land_surface_condition', 'foundation_type', 'roof_type',
                'ground_floor_type', 'other_floor_type', 'position',
                'plan_configuration', 'legal_ownership_status']

COL_GEO     = ['geo_level_1_id', 'geo_level_2_id', 'geo_level_3_id']

# Flag binari (has_)
COL_FLAG    = [c for c in train.columns if c.startswith('has_')]

COL_NUM     = ['count_floors_pre_eq', 'age', 'area_percentage',
                'height_percentage', 'count_families']

---
## 2. Missing Values

In [29]:
def missing_report(df, name='DataFrame'):
    missing = df.isna().sum()
    missing = missing[missing > 0]
    
    if missing.empty:
        print(f'{name}: nessun missing value!')
        return
    
missing_report(train, 'TRAIN')
missing_report(test_values, 'TEST')

TRAIN: nessun missing value!
TEST: nessun missing value!


---
## 3. Duplicati

In [46]:
duplicati = pd.DataFrame([
    {'Dataset': 'TRAIN', 'Righe Intere': train_values.duplicated().sum(), 'Solo ID': train_values['building_id'].duplicated().sum()},
    {'Dataset': 'TEST',  'Righe Intere': test_values.duplicated().sum(),  'Solo ID': test_values['building_id'].duplicated().sum()}
]).set_index('Dataset')

display(duplicati)

# Controllo Data Leakage (sovrapposizione ID tra Train e Test)
overlap = set(train_values['building_id']) & set(test_values['building_id'])

if not overlap:
    print('\n Nessuna sovrapposizione di ID tra TRAIN e TEST')
else:
    print(f'\n ATTENZIONE: {len(overlap)} ID in comune!')

,Righe Intere,Solo ID
Dataset,,
TRAIN,0,0
TEST,0,0



 Nessuna sovrapposizione di ID tra TRAIN e TEST


---
## 4. Cardinalità delle feature categoriche

In [ ]:
# Cardinalità categoriche
print(' Cardinalità e distribuzioni \n')
for col in COL_CAT:
    vc = train[col].value_counts()
    print(f'  {col:30s}  ({len(vc)} categorie)')
    for val, cnt in vc.items():
        print(f'      {val:15s}  {cnt:7d}  ({cnt/len(train)*100:.1f}%)')
    print()

=== Cardinalità e distribuzioni ===

  land_surface_condition          (3 categorie)
      t                 216757  (83.2%)
      n                  35528  (13.6%)
      o                   8316  (3.2%)

  foundation_type                 (5 categorie)
      r                 219196  (84.1%)
      w                  15118  (5.8%)
      u                  14260  (5.5%)
      i                  10579  (4.1%)
      h                   1448  (0.6%)

  roof_type                       (3 categorie)
      n                 182842  (70.2%)
      q                  61576  (23.6%)
      x                  16183  (6.2%)

  ground_floor_type               (5 categorie)
      f                 209619  (80.4%)
      x                  24877  (9.5%)
      v                  24593  (9.4%)
      z                   1004  (0.4%)
      m                    508  (0.2%)

  other_floor_type                (4 categorie)
      q                 165282  (63.4%)
      x                  43448  (16.7%)
      j  

---
## 5. Outlier e valori anomali — Variabili numeriche

In [22]:
# Statistiche descrittive numeriche
desc = train[COL_NUM].describe()
print(desc.to_string())

       count_floors_pre_eq            age  area_percentage  height_percentage  count_families
count        260601.000000  260601.000000    260601.000000      260601.000000   260601.000000
mean              2.129723      26.535029         8.018051           5.434365        0.983949
std               0.727665      73.565937         4.392231           1.918418        0.418389
min               1.000000       0.000000         1.000000           2.000000        0.000000
25%               2.000000      10.000000         5.000000           4.000000        1.000000
50%               2.000000      15.000000         7.000000           5.000000        1.000000
75%               2.000000      30.000000         9.000000           6.000000        1.000000
max               9.000000     995.000000       100.000000          32.000000        9.000000


In [12]:
df_num = train[COL_NUM]

Q1 = df_num.quantile(0.25)
Q3 = df_num.quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

# Maschera booleana per gli outlier (True se fuori dai limiti)
is_outlier = (df_num < lower) | (df_num > upper)

outlier_summary = pd.DataFrame({
    'Lower Bound': lower.round(1),
    'Upper Bound': upper.round(1),
    'Outliers': is_outlier.sum(),
    '%': (is_outlier.mean() * 100).round(2)
})

display(outlier_summary)

,Lower Bound,Upper Bound,Outliers,%
count_floors_pre_eq,2.0,2.0,103978,39.90
age,-20.0,60.0,12499,4.80
area_percentage,-1.0,15.0,13557,5.20
height_percentage,1.0,9.0,7843,3.01
count_families,1.0,1.0,34486,13.23


---
## 6. Valori particolari e sentinel values

In [ ]:
num_df = train[COL_NUM]

report = pd.DataFrame({
'Valori Mancanti (%)': (num_df.isna().mean() * 100).round(2),
'Zeri (%)': ((num_df == 0).mean() * 100).round(2),
'Minimo': num_df.min(),
'Massimo': num_df.max(),
'99° Percentile': num_df.quantile(0.99).round(2) # Utile per scovare le anomalie (es. age 995)
})

# Colonne dove il Massimo è molto più alto del 99° percentile (Outlier sospetti)
report['Outlier Sospetti'] = np.where(report['Massimo'] > (report['99° Percentile'] * 1.5), 'Si', 'No')
display(report)

,Valori Mancanti (%),Zeri (%),Minimo,Massimo,99° Percentile,Outlier Sospetti
count_floors_pre_eq,0.0,0.00,1,9,4.0,Si
age,0.0,9.99,0,995,100.0,Si
area_percentage,0.0,0.00,1,100,23.0,Si
height_percentage,0.0,0.00,2,32,11.0,Si
count_families,0.0,8.01,0,9,2.0,Si


---
## 7. Distribuzione del target

In [9]:
dist = pd.DataFrame({
    'Conteggio': train['damage_grade'].value_counts(),
    'Percentuale %': train['damage_grade'].value_counts(normalize=True) * 100
}).sort_index()

print("Distribuzione Target — damage_grade: \n")
print(dist)

Distribuzione Target — damage_grade: 

              Conteggio  Percentuale %
damage_grade                          
1                 25124       9.640792
2                148259      56.891186
3                 87218      33.468022


---
## 8. Feature geografiche — cardinalità

In [11]:
print('Cardinalità feature geografiche \n')
for col in COL_GEO:
    n = train[col].nunique()
    print(f'  {col:20s}  {n:5d} valori unici  '
          f'(min={train[col].min()}, max={train[col].max()})')

Cardinalità feature geografiche 

  geo_level_1_id           31 valori unici  (min=0, max=30)
  geo_level_2_id         1414 valori unici  (min=0, max=1427)
  geo_level_3_id        11595 valori unici  (min=0, max=12567)
